In [ ]:
import pandas as pd

# Cambia el nombre del archivo según corresponda
df = pd.read_csv('/content/drive/MyDrive/ING.DATOS/vehicles.csv')

print("--- Diagnóstico Básico Capa RAW ---")
print("Total de filas y columnas:", df.shape)
print("\nConteo de valores nulos por columna:\n", df.isnull().sum())
print("\nTotal de filas duplicadas exactas:", df.duplicated().sum())
print("\nTipos de datos detectados:\n", df.dtypes)

--- Diagnóstico Básico Capa RAW ---
Total de filas y columnas: (426880, 26)

Conteo de valores nulos por columna:
 id                   0
url                  0
region               0
region_url           0
price                0
year              1205
manufacturer     17646
model             5277
condition       174104
cylinders       177678
fuel              3013
odometer          4400
title_status      8242
transmission      2556
VIN             161042
drive           130567
size            306361
type             92858
paint_color     130203
image_url           68
description         70
county          426880
state                0
lat               6549
long              6549
posting_date        68
dtype: int64

Total de filas duplicadas exactas: 0

Tipos de datos detectados:
 id                int64
url              object
region           object
region_url       object
price             int64
year            float64
manufacturer     object
model            object
condition      

In [ ]:
import pandas as pd

# Cambia el nombre del archivo según corresponda
df = pd.read_csv('/content/drive/MyDrive/ING.DATOS/car_sales_data_2018_10.csv')

print("--- Diagnóstico Básico Capa RAW ---")
print("Total de filas y columnas:", df.shape)
print("\nConteo de valores nulos por columna:\n", df.isnull().sum())
print("\nTotal de filas duplicadas exactas:", df.duplicated().sum())
print("\nTipos de datos detectados:\n", df.dtypes)

--- Diagnóstico Básico Capa RAW ---
Total de filas y columnas: (211973, 9)

Conteo de valores nulos por columna:
 Date                 0
Salesperson          0
Customer Name        0
Car Make             0
Car Model            0
Car Year             0
Sale Price           0
Commission Rate      0
Commission Earned    0
dtype: int64

Total de filas duplicadas exactas: 0

Tipos de datos detectados:
 Date                  object
Salesperson           object
Customer Name         object
Car Make              object
Car Model             object
Car Year               int64
Sale Price             int64
Commission Rate      float64
Commission Earned    float64
dtype: object


In [ ]:
import pandas as pd

# Cambia el nombre del archivo según corresponda
df = pd.read_csv('/content/drive/MyDrive/ING.DATOS/all_regions.csv')

print("--- Diagnóstico Básico Capa RAW ---")
print("Total de filas y columnas:", df.shape)
print("\nConteo de valores nulos por columna:\n", df.isnull().sum())
print("\nTotal de filas duplicadas exactas:", df.duplicated().sum())
print("\nTipos de datos detectados:\n", df.dtypes)

--- Diagnóstico Básico Capa RAW ---
Total de filas y columnas: (1294757, 18)

Conteo de valores nulos por columna:
 brand                        0
name                         0
bodyType                     0
color                    37728
fuelType                  4942
year                    570113
mileage                 522958
transmission              5194
power                    21404
price                        0
vehicleConfiguration    570110
engineName              573781
engineDisplacement      577132
date                         0
location                     0
link                         0
description              40352
parse_date                   0
dtype: int64

Total de filas duplicadas exactas: 0

Tipos de datos detectados:
 brand                    object
name                     object
bodyType                 object
color                    object
fuelType                 object
year                    float64
mileage                 float64
transmission          

In [ ]:
# --- SCRIPT DDL PARA POSTGRESQL (Evidencia de Arquitectura) ---
# Este script se ejecutará en la base de datos final para crear el esquema estrella.

script_ddl = """
CREATE TABLE dim_vehiculo (
    id_vehiculo SERIAL PRIMARY KEY,
    marca VARCHAR(100) NOT NULL,
    modelo VARCHAR(100),
    anio_fabricacion INT,
    estado_condicion VARCHAR(50),
    tipo_carroceria VARCHAR(100),
    transmision VARCHAR(50)
);

CREATE TABLE dim_tiempo (
    id_tiempo INT PRIMARY KEY,
    fecha_completa DATE NOT NULL,
    anio INT,
    mes INT
);

CREATE TABLE fact_eventos_vehiculos (
    id_evento SERIAL PRIMARY KEY,
    id_vehiculo INT REFERENCES dim_vehiculo(id_vehiculo),
    id_tiempo INT REFERENCES dim_tiempo(id_tiempo),
    precio NUMERIC NOT NULL,
    kilometraje NUMERIC
);
"""
print("Script DDL documentado exitosamente. Listo para ingesta en PostgreSQL.")

Script DDL documentado exitosamente. Listo para ingesta en PostgreSQL.


In [ ]:
import pandas as pd

# 1. Leer el archivo más limpio desde la capa RAW
df = pd.read_csv('/content/drive/MyDrive/ING.DATOS/car_sales_data_2018_10.csv')

print("--- EXTRACCIÓN Y TRANSFORMACIÓN: DIMENSIÓN VEHÍCULO ---")

# 2. Seleccionar solo las columnas que describen físicamente al vehículo
columnas_vehiculo = df[['Car Make', 'Car Model', 'Car Year']]

# 3. Eliminar duplicados para crear un catálogo único
# (Un "Ford Mustang 2018" puede haberse vendido 50 veces, pero en nuestra dimensión solo debe existir una vez)
dim_vehiculo = columnas_vehiculo.drop_duplicates().reset_index(drop=True)

# 4. Renombrar las columnas para cumplir con el Contrato de Datos y el Diseño Dimensional
dim_vehiculo = dim_vehiculo.rename(columns={
    'Car Make': 'marca',
    'Car Model': 'modelo',
    'Car Year': 'anio_fabricacion'
})

# 5. Generar la Llave Primaria (id_vehiculo) para conectarla con la Tabla de Hechos
dim_vehiculo.insert(0, 'id_vehiculo', range(1, 1 + len(dim_vehiculo)))

# Mostrar la evidencia de transformación al profesor
print(f"Total de vehículos únicos catalogados a partir de las transacciones: {len(dim_vehiculo)}")
print("\nMuestra de la Tabla 'dim_vehiculo' lista para ingresar a la Capa Analítica:")
print(dim_vehiculo.head(10))

--- EXTRACCIÓN Y TRANSFORMACIÓN: DIMENSIÓN VEHÍCULO ---
Total de vehículos únicos catalogados a partir de las transacciones: 325

Muestra de la Tabla 'dim_vehiculo' lista para ingresar a la Capa Analítica:
   id_vehiculo      marca     modelo  anio_fabricacion
0            1     Toyota      Civic              2013
1            2      Honda      Civic              2022
2            3       Ford      Civic              2020
3            4     Toyota     Altima              2012
4            5       Ford  Silverado              2014
5            6     Nissan     Altima              2019
6            7  Chevrolet      F-150              2010
7            8      Honda     Altima              2012
8            9      Honda    Corolla              2014
9           10       Ford  Silverado              2011


In [ ]:
import pandas as pd

archivo_origen = '/content/drive/MyDrive/ING.DATOS/all_regions.csv'
archivo_destino = '/content/drive/MyDrive/ING.DATOS/all_regions_ing.csv'

# 'windows-1251' lee el cirílico ruso antiguo
# 'utf-8-sig' guarda en UTF-8 con BOM para que Excel inglés lo abra perfecto
chunk_size = 100000  # Procesa de 100 mil en 100 mil filas
first_chunk = True

for chunk in pd.read_csv(archivo_origen, encoding='utf-8', chunksize=chunk_size, low_memory=False):
    chunk.to_csv(archivo_destino, mode='a', encoding='utf-8-sig', index=False, header=first_chunk)
    first_chunk = False

print("¡Conversión masiva completada con éxito!")

¡Conversión masiva completada con éxito!
